# Yield Regression and Figures

This notebook runs first-pass yield regressions and saves plot outputs for long-term and acute experiments.

All paths are resolved relative to the project root.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import erf, sqrt

# Resolve project root whether notebook runs from repo root or notebooks/
ROOT = Path.cwd().resolve()
if not (ROOT / "cleaned_data").exists() and (ROOT.parent / "cleaned_data").exists():
    ROOT = ROOT.parent

CLEAN_DIR = ROOT / "cleaned_data"
FIG_DIR = CLEAN_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

LT_PATH = CLEAN_DIR / "longterm_yield_analysis_locked.csv"
ACUTE_PATH = CLEAN_DIR / "acute_yield_analysis_locked.csv"

print("ROOT:", ROOT)
print("LT_PATH exists:", LT_PATH.exists())
print("ACUTE_PATH exists:", ACUTE_PATH.exists())


In [ ]:
lt = pd.read_csv(LT_PATH)
acute = pd.read_csv(ACUTE_PATH)

lt["include_in_yield_model"] = lt["include_in_yield_model"].astype(int)
acute["include_in_yield_model"] = acute["include_in_yield_model"].astype(int)

lt_keep = lt[lt["include_in_yield_model"] == 1].copy()
acute_keep = acute[acute["include_in_yield_model"] == 1].copy()

# Long-term paired differences within cultivar x set
lt_wide = lt_keep.pivot(index=["cultivar", "set_id"], columns="heat_trt", values=[
    "healthy_weight_g", "pct_rotten_count", "pct_rotten_weight"
]).reset_index()
lt_wide.columns = [f"{a}_{b.lower()}" if isinstance(a, str) and b else a for a, b in lt_wide.columns]
lt_wide = lt_wide.rename(columns={
    "healthy_weight_g_control": "healthy_weight_control",
    "healthy_weight_g_otc": "healthy_weight_heat",
    "pct_rotten_count_control": "pct_rotten_count_control",
    "pct_rotten_count_otc": "pct_rotten_count_heat",
    "pct_rotten_weight_control": "pct_rotten_weight_control",
    "pct_rotten_weight_otc": "pct_rotten_weight_heat",
})
lt_wide["healthy_weight_diff"] = lt_wide["healthy_weight_heat"] - lt_wide["healthy_weight_control"]
lt_wide["pct_rotten_count_diff"] = lt_wide["pct_rotten_count_heat"] - lt_wide["pct_rotten_count_control"]
lt_wide["pct_rotten_weight_diff"] = lt_wide["pct_rotten_weight_heat"] - lt_wide["pct_rotten_weight_control"]

# Acute paired differences within cultivar x heat_level x replicate
acute_wide = acute_keep.pivot(index=["cultivar", "heat_level", "replicate"], columns="is_control", values=[
    "healthy_weight_g", "pct_rotten_count", "pct_rotten_weight"
]).reset_index()
acute_wide.columns = [
    f"{a}_{'control' if int(b)==1 else 'heat'}" if isinstance(a, str) and b in [0, 1] else a
    for a, b in acute_wide.columns
]
acute_wide["healthy_weight_diff"] = acute_wide["healthy_weight_g_heat"] - acute_wide["healthy_weight_g_control"]
acute_wide["pct_rotten_count_diff"] = acute_wide["pct_rotten_count_heat"] - acute_wide["pct_rotten_count_control"]
acute_wide["pct_rotten_weight_diff"] = acute_wide["pct_rotten_weight_heat"] - acute_wide["pct_rotten_weight_control"]

print("Long-term pairs:", len(lt_wide))
print("Acute pairs:", len(acute_wide))
lt_wide.head()


In [ ]:
def normal_two_tailed_p(t_value: float) -> float:
    # Normal approximation for two-tailed p-value (good for quick first-pass inference)
    z = abs(float(t_value))
    cdf = 0.5 * (1.0 + erf(z / sqrt(2.0)))
    return max(0.0, min(1.0, 2.0 * (1.0 - cdf)))


def fit_ols(X: np.ndarray, y: np.ndarray, term_names: list[str]) -> pd.DataFrame:
    n, p = X.shape
    xtx_inv = np.linalg.pinv(X.T @ X)
    beta = xtx_inv @ X.T @ y
    resid = y - X @ beta
    dof = max(n - p, 1)
    sigma2 = float((resid @ resid) / dof)
    se = np.sqrt(np.diag(xtx_inv) * sigma2)
    tvals = np.where(se > 0, beta / se, np.nan)
    pvals = [normal_two_tailed_p(t) if np.isfinite(t) else np.nan for t in tvals]
    return pd.DataFrame({
        "term": term_names,
        "estimate": beta,
        "std_error": se,
        "t_value": tvals,
        "p_value_normal_approx": pvals,
        "n_obs": n,
        "resid_variance": sigma2,
    })


def longterm_design(df: pd.DataFrame, outcome: str):
    y = df[outcome].to_numpy(float)
    st = (df["cultivar"] == "St").astype(float).to_numpy()
    X = np.column_stack([np.ones(len(df)), st])
    terms = ["Intercept(MQ)", "Cultivar_St"]
    return X, y, terms


def acute_design(df: pd.DataFrame, outcome: str):
    y = df[outcome].to_numpy(float)
    heat = pd.Categorical(df["heat_level"], categories=["A", "B", "C", "D"])
    st = (df["cultivar"] == "St").astype(float).to_numpy()
    hB = (heat == "B").astype(float)
    hC = (heat == "C").astype(float)
    hD = (heat == "D").astype(float)
    X = np.column_stack([
        np.ones(len(df)), hB, hC, hD, st,
        hB * st, hC * st, hD * st
    ])
    terms = [
        "Intercept(MQ,A)", "Heat_B", "Heat_C", "Heat_D", "Cultivar_St",
        "Heat_B_x_St", "Heat_C_x_St", "Heat_D_x_St"
    ]
    return X, y, terms

lt_outcomes = ["healthy_weight_diff", "pct_rotten_count_diff", "pct_rotten_weight_diff"]
acute_outcomes = ["healthy_weight_diff", "pct_rotten_count_diff", "pct_rotten_weight_diff"]

lt_coef_tables = []
for out in lt_outcomes:
    X, y, terms = longterm_design(lt_wide, out)
    t = fit_ols(X, y, terms)
    t.insert(0, "outcome", out)
    lt_coef_tables.append(t)
lt_coef = pd.concat(lt_coef_tables, ignore_index=True)

acute_coef_tables = []
for out in acute_outcomes:
    X, y, terms = acute_design(acute_wide, out)
    t = fit_ols(X, y, terms)
    t.insert(0, "outcome", out)
    acute_coef_tables.append(t)
acute_coef = pd.concat(acute_coef_tables, ignore_index=True)

lt_coef_path = CLEAN_DIR / "yield_longterm_model_coefficients_ipynb.csv"
acute_coef_path = CLEAN_DIR / "yield_acute_model_coefficients_ipynb.csv"
lt_coef.to_csv(lt_coef_path, index=False)
acute_coef.to_csv(acute_coef_path, index=False)

print("Saved:", lt_coef_path)
print("Saved:", acute_coef_path)
lt_coef.head(8)


In [ ]:
# Figure 1: Long-term healthy weight diff by cultivar
fig1, ax1 = plt.subplots(figsize=(7, 4))
for i, c in enumerate(["MQ", "St"]):
    vals = lt_wide.loc[lt_wide["cultivar"] == c, "healthy_weight_diff"].to_numpy()
    x = np.full(len(vals), i + 1)
    ax1.scatter(x, vals, alpha=0.8, label=c)
    ax1.hlines(vals.mean(), i + 0.75, i + 1.25, linewidth=3)
ax1.axhline(0, color="black", linewidth=1)
ax1.set_xticks([1, 2])
ax1.set_xticklabels(["MQ", "St"])
ax1.set_ylabel("Healthy weight diff (heat - control, g)")
ax1.set_title("Long-term: Healthy Weight Paired Differences")
ax1.grid(alpha=0.2)
fig1.tight_layout()
fig1_path = FIG_DIR / "yield_longterm_healthy_weight_diff.png"
fig1.savefig(fig1_path, dpi=180)
plt.show()

# Figure 2: Long-term rotten-count proportion diff by cultivar
fig2, ax2 = plt.subplots(figsize=(7, 4))
for i, c in enumerate(["MQ", "St"]):
    vals = lt_wide.loc[lt_wide["cultivar"] == c, "pct_rotten_count_diff"].to_numpy()
    x = np.full(len(vals), i + 1)
    ax2.scatter(x, vals, alpha=0.8, label=c)
    ax2.hlines(vals.mean(), i + 0.75, i + 1.25, linewidth=3)
ax2.axhline(0, color="black", linewidth=1)
ax2.set_xticks([1, 2])
ax2.set_xticklabels(["MQ", "St"])
ax2.set_ylabel("Rotten-count proportion diff (heat - control)")
ax2.set_title("Long-term: Rotten Proportion Paired Differences")
ax2.grid(alpha=0.2)
fig2.tight_layout()
fig2_path = FIG_DIR / "yield_longterm_rotten_count_diff.png"
fig2.savefig(fig2_path, dpi=180)
plt.show()

# Figure 3: Acute healthy weight diff by heat level and cultivar
heat_levels = ["A", "B", "C", "D"]
fig3, ax3 = plt.subplots(figsize=(9, 4.5))
positions = []
data = []
labels = []
for i, h in enumerate(heat_levels):
    for j, c in enumerate(["MQ", "St"]):
        vals = acute_wide.loc[(acute_wide["heat_level"] == h) & (acute_wide["cultivar"] == c), "healthy_weight_diff"].to_numpy()
        positions.append(i * 3 + j + 1)
        data.append(vals)
        labels.append(f"{h}-{c}")
ax3.boxplot(data, positions=positions, widths=0.65, patch_artist=False)
ax3.axhline(0, color="black", linewidth=1)
ax3.set_xticks(positions)
ax3.set_xticklabels(labels, rotation=45, ha="right")
ax3.set_ylabel("Healthy weight diff (heat - control, g)")
ax3.set_title("Acute: Healthy Weight Paired Differences")
ax3.grid(alpha=0.2)
fig3.tight_layout()
fig3_path = FIG_DIR / "yield_acute_healthy_weight_diff_boxplot.png"
fig3.savefig(fig3_path, dpi=180)
plt.show()

# Figure 4: Acute rotten-count proportion diff by heat level and cultivar
fig4, ax4 = plt.subplots(figsize=(9, 4.5))
positions2 = []
data2 = []
labels2 = []
for i, h in enumerate(heat_levels):
    for j, c in enumerate(["MQ", "St"]):
        vals = acute_wide.loc[(acute_wide["heat_level"] == h) & (acute_wide["cultivar"] == c), "pct_rotten_count_diff"].to_numpy()
        positions2.append(i * 3 + j + 1)
        data2.append(vals)
        labels2.append(f"{h}-{c}")
ax4.boxplot(data2, positions=positions2, widths=0.65, patch_artist=False)
ax4.axhline(0, color="black", linewidth=1)
ax4.set_xticks(positions2)
ax4.set_xticklabels(labels2, rotation=45, ha="right")
ax4.set_ylabel("Rotten-count proportion diff (heat - control)")
ax4.set_title("Acute: Rotten Proportion Paired Differences")
ax4.grid(alpha=0.2)
fig4.tight_layout()
fig4_path = FIG_DIR / "yield_acute_rotten_count_diff_boxplot.png"
fig4.savefig(fig4_path, dpi=180)
plt.show()

print("Saved figures:")
for p in [fig1_path, fig2_path, fig3_path, fig4_path]:
    print("-", p)


In [ ]:
summary_path = CLEAN_DIR / "yield_analysis_notebook_summary.md"
lines = [
    "# Yield Notebook Outputs",
    "",
    "This summary was generated from `notebooks/yield_regression_plots.ipynb`.",
    "",
    f"- Long-term pairs: {len(lt_wide)}",
    f"- Acute pairs: {len(acute_wide)}",
    f"- Long-term coefficient file: {(CLEAN_DIR / 'yield_longterm_model_coefficients_ipynb.csv')}",
    f"- Acute coefficient file: {(CLEAN_DIR / 'yield_acute_model_coefficients_ipynb.csv')}",
    f"- Figure directory: {FIG_DIR}",
]
summary_path.write_text("\n".join(lines), encoding="utf-8")
print("Saved:", summary_path)
summary_path
